# Prediction Market Data Profile

This notebook is a small local prototype for retrieving and comparing prediction-market data from Polymarket and Kalshi. It collects a narrow snapshot only: active/open markets, current orderbooks, and recent REST trades. It does not use WebSockets, authenticated trading endpoints, AWS, a CLI, or an API service.

The goal is to understand what each platform returns, preserve the raw payloads, and normalize the data into three comparable tables: `markets`, `orderbook_snapshots`, and `trades`.

## Source APIs and identifiers

Polymarket market discovery uses the public Gamma API. Orderbooks use public CLOB read endpoints, and recent market trades use the public Data API. A Polymarket market is usually tracked by `conditionId`, while orderbook snapshots are requested by outcome token IDs from `clobTokenIds`.

Kalshi market discovery, orderbooks, and trades use the public Trade API under `https://external-api.kalshi.com/trade-api/v2`. A Kalshi market is tracked by `ticker`.

Both platforms expose binary Yes/No market data, but their orderbook shapes differ. Polymarket has a CLOB book per outcome token. Kalshi returns Yes and No bid ladders; a No bid at price `X` implies a Yes ask at `1 - X`, and a Yes bid at `Y` implies a No ask at `1 - Y`.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from prediction_market.collectors import collect_snapshot
from prediction_market.storage import save_snapshot

DATA_DIR = PROJECT_ROOT / "data"
MARKET_LIMIT = 10
TRADES_LIMIT = 100
ORDERBOOK_DEPTH = 100

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

## Retrieve and persist the snapshot

This cell makes live unauthenticated requests. It writes raw JSON first, then normalized Parquet and CSV files. Re-running the cell overwrites the previous local snapshot files.

In [2]:
raw_data = collect_snapshot(
    market_limit=MARKET_LIMIT,
    trades_limit=TRADES_LIMIT,
    orderbook_depth=ORDERBOOK_DEPTH,
)
saved = save_snapshot(raw_data, DATA_DIR)
tables = saved["tables"]

print(f"Snapshot retrieved at: {raw_data['retrieved_at']}")
print(f"Polymarket markets: {len(raw_data['polymarket_markets'])}")
print(f"Kalshi markets: {len(raw_data['kalshi_markets'])}")
print(f"Polymarket orderbook token payloads: {len(raw_data['polymarket_orderbooks'])}")
print(f"Kalshi orderbook payloads: {len(raw_data['kalshi_orderbooks'])}")
print(f"Polymarket trade groups: {len(raw_data['polymarket_trades'])}")
print(f"Kalshi trade groups: {len(raw_data['kalshi_trades'])}")

Snapshot retrieved at: 2026-06-06T19:51:13Z
Polymarket markets: 10
Kalshi markets: 10
Polymarket orderbook token payloads: 20
Kalshi orderbook payloads: 10
Polymarket trade groups: 10
Kalshi trade groups: 10


In [3]:
print("Raw files:")
for name, path in saved["raw_paths"].items():
    print(f"  {name}: {path.relative_to(PROJECT_ROOT)}")

print("\nProcessed files:")
for table_name, paths in saved["processed_paths"].items():
    print(f"  {table_name}: {paths['parquet'].relative_to(PROJECT_ROOT)} and {paths['csv'].relative_to(PROJECT_ROOT)}")

Raw files:
  snapshot_metadata: data/raw/snapshot_metadata.json
  polymarket_markets: data/raw/polymarket_markets.json
  kalshi_markets: data/raw/kalshi_markets.json
  polymarket_orderbooks: data/raw/polymarket_orderbooks.json
  kalshi_orderbooks: data/raw/kalshi_orderbooks.json
  polymarket_trades: data/raw/polymarket_trades.json
  kalshi_trades: data/raw/kalshi_trades.json

Processed files:
  markets: data/processed/markets.parquet and data/processed/markets.csv
  orderbook_snapshots: data/processed/orderbook_snapshots.parquet and data/processed/orderbook_snapshots.csv
  trades: data/processed/trades.parquet and data/processed/trades.csv


## Raw payload samples

These examples show why the normalization layer is intentionally small. The raw payloads include many platform-specific fields, and those fields can differ even when the common market concept is similar.

In [4]:
def show_json(label, value, max_chars=5000):
    print(f"\n--- {label} ---")
    if value is None:
        print("No payload available")
        return
    rendered = json.dumps(value, indent=2, sort_keys=True, default=str)
    print(rendered[:max_chars])
    if len(rendered) > max_chars:
        print("... truncated ...")

show_json("Polymarket market", raw_data["polymarket_markets"][0] if raw_data["polymarket_markets"] else None)
show_json("Kalshi market", raw_data["kalshi_markets"][0] if raw_data["kalshi_markets"] else None)
show_json("Polymarket orderbook token payload", raw_data["polymarket_orderbooks"][0] if raw_data["polymarket_orderbooks"] else None)
show_json("Kalshi orderbook payload", raw_data["kalshi_orderbooks"][0] if raw_data["kalshi_orderbooks"] else None)
show_json("Polymarket trade group", raw_data["polymarket_trades"][0] if raw_data["polymarket_trades"] else None)
show_json("Kalshi trade group", raw_data["kalshi_trades"][0] if raw_data["kalshi_trades"] else None)


--- Polymarket market ---
{
  "acceptingOrders": true,
  "acceptingOrdersTimestamp": "2025-05-02T15:47:45Z",
  "active": true,
  "approved": true,
  "archived": false,
  "automaticallyActive": true,
  "bestAsk": 0.53,
  "bestBid": 0.51,
  "clearBookOnStart": true,
  "clobTokenIds": "[\"98022490269692409998126496127597032490334070080325855126491859374983463996227\", \"53831553061883006530739877284105938919721408776239639687877978808906551086026\"]",
  "closed": false,
  "competitive": 0.9996001599360256,
  "conditionId": "0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be",
  "createdAt": "2025-05-02T15:04:43.762151Z",
  "customLiveness": 0,
  "cyom": false,
  "deploying": false,
  "deployingTimestamp": "2025-05-02T15:47:04.727292Z",
  "description": "This market will resolve to \"Yes\" if Rihanna officially releases a new album before Grand Theft Auto VI is officially released in the US. Otherwise, this market will resolve to \"No\".\n\nIf neither occurs by July 31, 20

## Normalized table: markets

The `markets` table keeps one row per selected market. The comparable columns are platform, market ID, ticker or slug, title, status, close time, outcomes, volume, and liquidity. The `raw_payload` column preserves the full original JSON so platform-specific fields are not lost.

Polymarket outcomes often arrive as JSON encoded strings, such as `["Yes", "No"]`; the helper code parses those into Python lists. Kalshi markets use the market ticker as the stable identifier and use `yes_sub_title` / `no_sub_title` to describe the two sides.

In [5]:
markets = tables["markets"]
display(markets.drop(columns=["raw_payload"]))

,platform,market_id,ticker_or_slug,title,category,status,close_time,outcomes,volume,liquidity
0,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,new-rhianna-album-before-gta-vi-926,New Rihanna Album before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",8.189309e+05,22380.57250
1,polymarket,0x50ddb9cd80d5c271664a2ebb7fcaed1d0a148d82c8e8d314d830f75a944c3dcc,new-playboi-carti-album-before-gta-vi-421,New Playboi Carti Album before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",7.389957e+05,9541.43100
2,polymarket,0x32b09f6390252b37d674501527e709016d55581b2c1e544bd4b8167f5f732f4c,will-jesus-christ-return-before-gta-vi-665,Will Jesus Christ return before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",1.154658e+07,480433.11210
3,polymarket,0x84f8b70331323c2fba97d7ceaa9a35fb645a0770d0dbff169d07f24f376766e9,trump-out-as-president-before-gta-vi-846,Trump out as President before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",6.635039e+05,28036.19240
4,polymarket,0x7b49b9bacb5f435bc10f3b100ff59e2fdd346f7f92a9001881bc9825a0af0f11,will-china-invades-taiwan-before-gta-vi-716-644,Will China invades Taiwan before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",1.860894e+06,25561.84740
5,polymarket,0xbb57ccf5853a85487bc3d83d04d669310d28c6c810758953b9d9b91d1aee89d2,will-bitcoin-hit-1m-before-gta-vi-872-424,Will bitcoin hit $1m before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",4.453722e+06,112531.01008
6,polymarket,0xf398b0e5016eeaee9b0885ed84012b6dc91269ac10d3b59d60722859c2e30b2f,will-harvey-weinstein-be-sentenced-to-no-prison-time,Will Harvey Weinstein be sentenced to no prison time?,None,active,2025-12-31T12:00:00Z,"[Yes, No]",3.733480e+05,1462.10423
7,polymarket,0xe2b48e3b44de9658ee9c8b37354301763e33c0b502fd966839d644b4c0a9dea8,will-harvey-weinstein-be-sentenced-to-less-than-5-years-in-prison,Will Harvey Weinstein be sentenced to less than 5 years in prison?,None,active,2025-12-31T12:00:00Z,"[Yes, No]",1.394779e+05,3572.94531
8,polymarket,0x3209617364a0d598435806b59d0d056b606022dc9028c466ad7912df94fc170c,will-harvey-weinstein-be-sentenced-to-between-5-and-10-years-in-prison,Will Harvey Weinstein be sentenced to between 5 and 10 years in prison?,None,active,2025-12-31T12:00:00Z,"[Yes, No]",8.905461e+04,2413.00294
9,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,will-harvey-weinstein-be-sentenced-to-between-10-and-20-years-in-prison,Will Harvey Weinstein be sentenced to between 10 and 20 years in prison?,None,active,2025-12-31T12:00:00Z,"[Yes, No]",1.693886e+05,2254.29894


In [6]:
display(
    markets.groupby("platform", dropna=False)
    .agg(
        market_count=("market_id", "count"),
        total_volume=("volume", "sum"),
        total_liquidity=("liquidity", "sum"),
    )
    .reset_index()
)

,platform,market_count,total_volume,total_liquidity
0,kalshi,10,0.000000e+00,0.0000
1,polymarket,10,2.085390e+07,688186.5169


## Normalized table: orderbook snapshots

The `orderbook_snapshots` table keeps one row per selected market. Prices are represented as decimal dollars/probabilities from `0.0` to `1.0`.

For Polymarket, the notebook requests each binary market's first two CLOB token IDs and treats token index 0 as Yes and token index 1 as No when the raw outcome text is not explicit. Best bids are the highest bid price and best asks are the lowest ask price from each token book.

For Kalshi, the API returns bid ladders only. The normalized Yes ask is calculated from the best No bid as `1 - best_no_bid`. The normalized No ask is calculated from the best Yes bid as `1 - best_yes_bid`.

In [7]:
orderbooks = tables["orderbook_snapshots"]
display(orderbooks.drop(columns=["depth_json"]))

,platform,market_id,timestamp,best_yes_bid,best_yes_ask,best_no_bid,best_no_ask,spread,bid_depth,ask_depth
0,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,2026-06-06T19:51:12Z,0.510,0.520,0.480,0.490,0.010,215123.28,47075.86
1,polymarket,0x50ddb9cd80d5c271664a2ebb7fcaed1d0a148d82c8e8d314d830f75a944c3dcc,2026-06-06T19:51:11Z,0.520,0.540,0.460,0.480,0.020,42897.72,34810.29
2,polymarket,0x32b09f6390252b37d674501527e709016d55581b2c1e544bd4b8167f5f732f4c,2026-06-06T19:50:55Z,0.490,0.500,0.500,0.510,0.010,1787644.91,1611576.48
3,polymarket,0x84f8b70331323c2fba97d7ceaa9a35fb645a0770d0dbff169d07f24f376766e9,2026-06-06T19:50:57Z,0.500,0.510,0.490,0.500,0.010,90467.85,66644.06
4,polymarket,0x7b49b9bacb5f435bc10f3b100ff59e2fdd346f7f92a9001881bc9825a0af0f11,2026-06-06T19:51:12Z,0.500,0.510,0.490,0.500,0.010,98117.67,93975.65
5,polymarket,0xbb57ccf5853a85487bc3d83d04d669310d28c6c810758953b9d9b91d1aee89d2,2026-06-06T19:50:48Z,0.492,0.493,0.507,0.508,0.001,233960.09,141915.29
6,polymarket,0xf398b0e5016eeaee9b0885ed84012b6dc91269ac10d3b59d60722859c2e30b2f,2026-06-06T19:51:12Z,0.773,0.838,0.162,0.227,0.065,20640.76,4418.86
7,polymarket,0xe2b48e3b44de9658ee9c8b37354301763e33c0b502fd966839d644b4c0a9dea8,2026-06-06T19:51:09Z,0.022,0.043,0.957,0.978,0.021,4330.28,10837.51
8,polymarket,0x3209617364a0d598435806b59d0d056b606022dc9028c466ad7912df94fc170c,2026-06-06T19:51:09Z,0.025,0.041,0.959,0.975,0.016,3093.57,11565.98
9,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,2026-06-06T19:50:09Z,0.023,0.087,0.913,0.977,0.064,4420.69,3682.89


In [8]:
spread_view = orderbooks.drop(columns=["depth_json"]).copy()
spread_view = spread_view.sort_values(["platform", "spread"], na_position="last")
display(spread_view[["platform", "market_id", "best_yes_bid", "best_yes_ask", "spread", "bid_depth", "ask_depth"]])

,platform,market_id,best_yes_bid,best_yes_ask,spread,bid_depth,ask_depth
10,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S2026DCDC33AFF62-394B92C7FBF,NaN,0.119,NaN,0.00,93.00
11,kalshi,KXMVECROSSCATEGORY-S20268CD92AFF091-90F94552E6D,NaN,0.026,NaN,0.00,832.00
12,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S2026573CBCEDB7A-67D257C4731,NaN,0.640,NaN,0.00,18.00
13,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S20266BD56DE9E46-B9B63E7A79A,NaN,NaN,NaN,0.00,0.00
14,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S20265A65C1BFAEE-48D23EA828E,NaN,0.039,NaN,0.00,355.00
15,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S20266BBC4DD92D5-04CD166327B,NaN,0.044,NaN,0.00,240.00
16,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S2026B03F2D5C908-3A2846429FD,NaN,NaN,NaN,0.00,0.00
17,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S202616199EFB802-865C910FF2A,NaN,0.076,NaN,0.00,145.00
18,kalshi,KXMVESPORTSMULTIGAMEEXTENDED-S202647BEB9308EF-5A4AB4CA956,NaN,NaN,NaN,0.00,0.00
19,kalshi,KXMVECROSSCATEGORY-S20260E8E35F300A-2CB1086CB9A,NaN,0.839,NaN,0.00,14.00


## Normalized table: trades

The `trades` table keeps one row per recent trade returned by the REST endpoints. Some active markets can have no recent trades; in that case the table can be empty for that market, which is expected and should not be treated as a collection failure.

Polymarket trade prices are attached to the traded outcome token. The normalized `yes_price` keeps the Yes-side probability: a No outcome trade at `0.47` becomes a Yes probability of `0.53`. Kalshi trade payloads already provide `yes_price_dollars`, so `price` and `yes_price` are the same for Kalshi rows.

In [9]:
trades = tables["trades"]
display(trades.drop(columns=["raw_payload"]))

if trades.empty:
    print("No recent trades were returned for the selected markets.")
else:
    display(
        trades.groupby(["platform", "market_id"], dropna=False)
        .agg(
            trade_count=("trade_id", "count"),
            latest_trade_time=("timestamp", "max"),
            avg_yes_price=("yes_price", "mean"),
        )
        .reset_index()
        .sort_values(["platform", "trade_count"], ascending=[True, False])
    )

,platform,market_id,trade_id,timestamp,outcome,side,price,yes_price,size
0,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,0x0453fdd51843a21143e819aa447d3082b4f1c71d4acb168cc44fa74b039ab927,2026-06-06T19:22:52Z,No,BUY,0.490,0.510,10.204080
1,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,0xa4893f1692b79635865042eb21be3506aa7e7b4c8c3cdbffda5cb0b0060aefe6,2026-06-06T19:19:25Z,No,SELL,0.470,0.530,6.220000
2,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,0xbe122da6fa7d258d4a72461e2a6c43d1a443a26969527f2f96a2602a5a0c9b3a,2026-06-06T18:46:52Z,No,BUY,0.490,0.510,6.224488
3,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,0xabc028a2583539d8920db8b0bb43f950a33c6fdaf8a5fc36e3bb5e8eb62f52eb,2026-06-06T18:20:45Z,Yes,SELL,0.510,0.510,5.190000
4,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,0x0283c61893c06e58323ab21deecdfc9efb5ed84d8d98a5ac2d7aee60fb2bdcf8,2026-06-06T18:20:04Z,Yes,BUY,0.530,0.530,5.188678
...,...,...,...,...,...,...,...,...,...
995,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,0x31a7ac3e6ff65190064c9784fec6cf818e2f5a5ede9120bfccf7bac470af457e,2026-05-27T18:56:47Z,No,SELL,0.923,0.077,10.000000
996,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,0x3718947f29d888020bfbb25243f4890494b5e693feac55cbd947001fa8a1184a,2026-05-27T15:33:21Z,Yes,SELL,0.062,0.062,2.660000
997,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,0x2b0dcc1af1a08c527065ab6827fa4699d6cd7dfe21f72c8da49bd1735db3ee23,2026-05-27T14:45:43Z,No,SELL,0.921,0.079,8.100000
998,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,0xc481fa48cbec54b42d5ddbfbd74b7a9b4877b872cbc27fcca8727cfecfcbcbb6,2026-05-27T14:45:10Z,No,BUY,0.930,0.070,8.100000


,platform,market_id,trade_count,latest_trade_time,avg_yes_price
0,polymarket,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1070987bd1a3316f772be,100,2026-06-06T19:22:52Z,0.511185
1,polymarket,0x3209617364a0d598435806b59d0d056b606022dc9028c466ad7912df94fc170c,100,2026-06-05T02:20:04Z,0.039719
2,polymarket,0x32b09f6390252b37d674501527e709016d55581b2c1e544bd4b8167f5f732f4c,100,2026-06-06T18:58:27Z,0.490700
3,polymarket,0x3d495a3e05eaffe438bb1c2ab97ed57a79b0a6ab18a2ca6fa5b448e20ce70082,100,2026-06-06T08:16:07Z,0.070649
4,polymarket,0x50ddb9cd80d5c271664a2ebb7fcaed1d0a148d82c8e8d314d830f75a944c3dcc,100,2026-06-06T18:12:27Z,0.524781
5,polymarket,0x7b49b9bacb5f435bc10f3b100ff59e2fdd346f7f92a9001881bc9825a0af0f11,100,2026-06-06T16:11:39Z,0.503500
6,polymarket,0x84f8b70331323c2fba97d7ceaa9a35fb645a0770d0dbff169d07f24f376766e9,100,2026-06-06T17:27:19Z,0.505400
7,polymarket,0xbb57ccf5853a85487bc3d83d04d669310d28c6c810758953b9d9b91d1aee89d2,100,2026-06-06T19:42:43Z,0.492640
8,polymarket,0xe2b48e3b44de9658ee9c8b37354301763e33c0b502fd966839d644b4c0a9dea8,100,2026-06-05T13:42:33Z,0.048626
9,polymarket,0xf398b0e5016eeaee9b0885ed84012b6dc91269ac10d3b59d60722859c2e30b2f,100,2026-06-06T11:40:13Z,0.757252


## Keyword comparison example

This lightweight comparison searches market titles and IDs across both platforms. Change `KEYWORD` to inspect another theme, such as `bitcoin`, `election`, `nba`, or `weather`.

In [10]:
KEYWORD = "bitcoin"

keyword_mask = (
    markets["title"].fillna("").str.contains(KEYWORD, case=False, regex=False)
    | markets["ticker_or_slug"].fillna("").str.contains(KEYWORD, case=False, regex=False)
)

keyword_markets = markets.loc[keyword_mask].drop(columns=["raw_payload"])
display(keyword_markets if not keyword_markets.empty else pd.DataFrame({"message": [f"No selected markets matched '{KEYWORD}'"]}))

,platform,market_id,ticker_or_slug,title,category,status,close_time,outcomes,volume,liquidity
5,polymarket,0xbb57ccf5853a85487bc3d83d04d669310d28c6c810758953b9d9b91d1aee89d2,will-bitcoin-hit-1m-before-gta-vi-872-424,Will bitcoin hit $1m before GTA VI?,None,active,2026-07-31T12:00:00Z,"[Yes, No]",4.453722e+06,112531.01008


## What this snapshot proves

This notebook proves the local ingestion path for both platforms: retrieve small live snapshots, preserve source JSON, normalize comparable market/orderbook/trade fields, and write analysis-friendly Parquet and CSV artifacts.

The main platform differences to carry forward are identifier shape, market metadata richness, orderbook representation, and trade price semantics. WebSocket collection, authenticated endpoints, historical backfills, and cloud deployment should stay as later phases after this snapshot workflow is stable.